In [12]:
import sys
print(sys.executable)


/home/prithwis/TrackSwipe/.venv/bin/python3


import cv2
cap = cv2.VideoCapture(0)
print("Camera opened:", cap.isOpened())
cap.release()


In [2]:
import cv2
from deepface import DeepFace
from collections import deque
from io import BytesIO
from IPython.display import display, clear_output
from PIL import Image
import numpy as np
import subprocess
import pandas as pd
import os
import ipywidgets as widgets
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import other_params  
import csv
import time
import re
from collections import defaultdict
import sounddevice as sd
import librosa
import threading

2025-11-09 08:04:22.178104: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-09 08:04:22.256085: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


('clear sky', 7.76)


In [ ]:
# -----------------------
# CONFIG
# -----------------------
CONFIDENCE_THRESHOLD = 0.4
NUM_FRAMES = 20
EMOTION_HISTORY_LEN = 10
OLLAMA_MODEL = "llama2"
CSV_FILE = "user_logs.csv"
emotion_history_with_confidence = deque(maxlen=EMOTION_HISTORY_LEN)
SAMPLE_RATE = 16000  # Hz
DURATION = 10        # seconds, match roughly the video capture duration

audio_data = None
# -----------------------
# SETUP CSV FILE
# -----------------------
if not os.path.exists(CSV_FILE):
    with open(CSV_FILE, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            "timestamp", "city", "latitude", "longitude", "today_status", "tomorrow_status",
            "weekday", "weather_desc", "temperature", "mood", "voice_tone", "movie_selected"
        ])
        
# -----------------------
# OLLAMA CALL
# -----------------------
def ask_ollama(prompt_text):
    """Call Ollama CLI to get movie recommendations"""
    try:
        result = subprocess.run(
            ["ollama", "run", OLLAMA_MODEL, prompt_text],
            capture_output=True, text=True, check=True
        )
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        print("Error calling Ollama:", e)
        return ""

# -----------------------
# LOCATION, WEATHER, TIME
# -----------------------
city, lat, lon = other_params.get_location()
today_status, tomorrow_status, weekday_name = other_params.today_and_tomorrow_status()
weather_desc, temperature = other_params.get_weather()
voice_tone = "neutral"

print(f"📍 Location: {city}")
print(f"📅 Today: {today_status} ({weekday_name}), Tomorrow: {tomorrow_status}")
print(f"🌤 Weather: {weather_desc}, {temperature}°C")
print("\nCapturing emotional context...")


def record_audio():
    global audio_data
    print("🎤 Recording audio...")
    audio_data = sd.rec(int(DURATION * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
    sd.wait()
    print("✅ Audio capture complete")
    audio_data = audio_data.flatten()

# Start audio recording in a separate thread
audio_thread = threading.Thread(target=record_audio)
audio_thread.start()


# -----------------------
# EMOTION CAPTURE (short burst)
# -----------------------
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

def preprocess_frame(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    enhanced = cv2.merge([l,a,b])
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)

#emotions = []
#audio_data = sd.rec(int(DURATION * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
#sd.wait()

for i in range(NUM_FRAMES):
    ret, frame = cap.read()
    if not ret:
        break
        
    enhanced_frame = preprocess_frame(frame)
    gray = cv2.cvtColor(enhanced_frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=7, minSize=(60,60))
    
    for (x, y, w, h) in faces:
        face_roi = enhanced_frame[y:y+h, x:x+w]
        try:
            # Analyze only emotion (fastest)
            result = DeepFace.analyze(enhanced_frame, actions=['emotion'], enforce_detection=False, detector_backend='opencv', silent=True)
            dominant_emotion = result[0]['dominant_emotion']
            confidence = result[0]['emotion'][dominant_emotion] / 100.0
            if confidence >= CONFIDENCE_THRESHOLD:
                #emotions.append(dominant_emotion)
                emotion_history_with_confidence.append((dominant_emotion, confidence))
            print(f"Frame {i+1}/{NUM_FRAMES}: {dominant_emotion}")
        except Exception as e:
            print(f"Skipped frame {i+1}: {e}")
            continue

    time.sleep(0.2)  # small delay between captures

cap.release()

# Wait for audio to finish recording
audio_thread.join()
print("Audio captured shape:", audio_data.shape)

def extract_audio_features(audio_data, sr=16000):
    audio_data = audio_data.astype(np.float32)
    audio_data = audio_data / (np.max(np.abs(audio_data)) + 1e-6)  # normalize

    # RMS energy
    rms = np.mean(librosa.feature.rms(y=audio_data))

    # Pitch
    pitches, magnitudes = librosa.piptrack(y=audio_data, sr=sr)
    pitch = np.mean(pitches[pitches>0]) if np.any(pitches>0) else 0

    # Zero crossing rate
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=audio_data))

    # Spectral centroid
    spec_centroid = np.mean(librosa.feature.spectral_centroid(y=audio_data, sr=sr))

    return rms, pitch, zcr, spec_centroid

def estimate_voice_emotion(audio_data, sr=16000):
    rms, pitch, zcr, spec_centroid = extract_audio_features(audio_data, sr)

    # Rules (tuned for Jetson mic)
    if rms > 0.02 and pitch > 120 and zcr > 0.02:
        voice_tone = "happy"
    elif rms < 0.01 and pitch < 100 and zcr < 0.01:
        voice_tone = "sad"
    else:
        voice_tone = "neutral"

    return voice_tone

voice_tone = estimate_voice_emotion(audio_data)

# -----------------------
# SMOOTH EMOTION
# -----------------------
def get_weighted_smoothed_emotion(emotion_history_with_confidence):
    """
    emotion_history_with_confidence: list of tuples [(emotion, confidence), ...]
    Returns: the weighted dominant emotion
    """
    if not emotion_history_with_confidence:
        return "neutral"
    weighted_scores = defaultdict(float)
    for emotion, conf in emotion_history_with_confidence:
        weighted_scores[emotion] += conf  # add confidence as weight
    # picking the emotion with highest total confidence
    dominant = max(weighted_scores, key=weighted_scores.get)
    return dominant
    
smoothed_emotion = get_weighted_smoothed_emotion(emotion_history_with_confidence)

print(f"\n🧠 Final detected mood: {smoothed_emotion}")

# -----------------------
# ASK OLLAMA
# -----------------------
prompt = f"""
You are a movie recommendation assistant. The user context is:
- Location: {city} ({lat:.2f}, {lon:.2f})
- Today: {today_status} ({weekday_name})
- Tomorrow: {tomorrow_status}
- Current weather: {weather_desc}, {temperature}°C
- Current mood: {smoothed_emotion}
- Voice tone: {voice_tone}
Recommend 5 movie names that the user is most likely to enjoy right now. 
"""
# Extract movie titles from response 
print("\n🎬 Fetching movie recommendations...")
recommendations = ask_ollama(prompt)
movie_titles = re.findall(r"\d+\.\s*(.*?)\s*\(\d{4}\)", recommendations)

# -----------------------
# TRAIN/UPDATE LEARNING MODEL
# -----------------------
def train_user_model(csv_file):
    if not os.path.exists(csv_file):
        return None, None
    
    df = pd.read_csv(csv_file)
    if df.empty:
        return None, None
    
    possible_emotions = ['angry','disgust','fear','happy','sad','surprise','neutral']

    le_city = LabelEncoder()
    le_today = LabelEncoder()
    le_mood = LabelEncoder()
    le_mood.fit(possible_emotions)   # keep all possible emotions to avoid crash
    le_movie = LabelEncoder()
    
    features = df[['latitude', 'longitude', 'temperature']].copy()
    features['city'] = le_city.fit_transform(df['city'])
    features['today_status'] = le_today.fit_transform(df['today_status'])
    features['mood'] = le_mood.transform(df['mood'])
    target = le_movie.fit_transform(df['movie_selected'])
    
    clf = RandomForestClassifier(n_estimators=50)
    clf.fit(features, target)
    return clf, (le_city, le_today, le_mood, le_movie)

def combined_recommendations(primary_movies, clf_tuple, user_context):
    clf, encoders = clf_tuple
    if clf is None:
        return primary_movies
    
    le_city, le_today, le_mood, le_movie = encoders
    df = pd.DataFrame([{
        'latitude': user_context['lat'],
        'longitude': user_context['lon'],
        'temperature': user_context['temperature'],
        'city': le_city.transform([user_context['city']])[0],
        'today_status': le_today.transform([user_context['today_status']])[0],
        'mood': le_mood.transform([user_context['mood']])[0]
    }])
    
    pred_probs = clf.predict_proba(df)[0]
    top_indices = pred_probs.argsort()[::-1]
    top_movies = le_movie.inverse_transform(top_indices)
    
    final_list = []
    for movie in top_movies:
        if movie in primary_movies and movie not in final_list:
            final_list.append(movie)
    for movie in primary_movies:
        if movie not in final_list:
            final_list.append(movie)
    
    return final_list[:5]

# -----------------------
# DISPLAY BUTTONS FOR USER CLICK
# -----------------------
buttons = []
output_box = widgets.Output()
def on_click(b):
    with output_box:
        clear_output(wait=True)
        print(f"🎬 You selected: {b.description}")
        # Disable all buttons
        for btn in buttons:
            btn.disabled = True
            
        # Log user feedback
        df = pd.DataFrame([{
            'timestamp': pd.Timestamp.now(),
            'city': city,
            'latitude': lat,
            'longitude': lon,
            'today_status': today_status,
            'tomorrow_status': tomorrow_status,
            'weekday': weekday_name,
            'weather_desc': weather_desc,
            'temperature': temperature,
            'mood': smoothed_emotion,
            'voice_tone': voice_tone,
            'movie_selected': b.description
        }])
        if os.path.exists(CSV_FILE):
            df.to_csv(CSV_FILE, mode='a', header=False, index=False)
        else:
            df.to_csv(CSV_FILE, index=False)
'''
for movie in movie_titles:
    btn = widgets.Button(
        description=movie,
        layout=widgets.Layout(width='auto', margin='5px')
    )
    btn.on_click(on_click)
    buttons.append(btn)

buttons_box = widgets.HBox(buttons)  # side-by-side
display(widgets.HTML("<h3>Select a Movie 🎥</h3>"))
display(buttons_box)
display(output_box)
'''
# -----------------------
# TRAIN LEARNING MODEL & COMBINE RECOMMENDATIONS
# -----------------------
clf_tuple = train_user_model(CSV_FILE)
user_context = {
    'city': city,
    'lat': lat,
    'lon': lon,
    'today_status': today_status,
    'temperature': temperature,
    'mood': smoothed_emotion
}
final_movies = combined_recommendations(movie_titles, clf_tuple, user_context)
#print("🎯 Final combined recommendations:", final_movies)
for movie in final_movies:
    btn = widgets.Button(
        description=movie,
        layout=widgets.Layout(width='auto', margin='5px')
    )
    btn.on_click(on_click)
    buttons.append(btn)
buttons_box = widgets.HBox(buttons)  # side-by-side
display(widgets.HTML("<h3>Select a Movie 🎥</h3>"))
display(buttons_box)
display(output_box)

📍 Location: Seoul
📅 Today: Weekend (Sunday), Tomorrow: Weekday
🌤 Weather: clear sky, 7.76°C

Capturing emotional context...
🎤 Recording audio...


2025-11-09 08:04:32.201187: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Frame 1/20: fear
Frame 1/20: fear
Frame 3/20: neutral
Frame 4/20: fear
Frame 5/20: neutral
Frame 8/20: neutral
Frame 9/20: neutral
Frame 10/20: neutral
Frame 11/20: neutral
Frame 12/20: neutral
Frame 13/20: neutral
Frame 14/20: fear
Frame 15/20: neutral
Frame 15/20: neutral
Frame 16/20: neutral
✅ Audio capture complete
Frame 17/20: neutral
Frame 18/20: neutral
Frame 18/20: neutral
Frame 19/20: happy
Frame 19/20: happy
Frame 20/20: happy
Audio captured shape: (160000,)

🧠 Final detected mood: neutral

🎬 Fetching movie recommendations...


In [28]:
import sounddevice as sd
import librosa
from sklearn.ensemble import RandomForestClassifier

SAMPLE_RATE = 16000  # Hz
DURATION = 10        # seconds, match roughly the video capture duration

# -----------------------
# RECORD AUDIO
# -----------------------
print("🎤 Recording audio...")
audio_data = sd.rec(int(DURATION * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
sd.wait()  # wait until recording is finished
print("✅ Audio capture complete")

# Convert to numpy array
audio_data = audio_data.flatten()
print("Audio shape:", audio_data.shape)

def extract_audio_features(audio_data, sr=16000):
    audio_data = audio_data.astype(np.float32)
    audio_data = audio_data / (np.max(np.abs(audio_data)) + 1e-6)  # normalize

    # RMS energy
    rms = np.mean(librosa.feature.rms(y=audio_data))

    # Pitch
    pitches, magnitudes = librosa.piptrack(y=audio_data, sr=sr)
    pitch = np.mean(pitches[pitches>0]) if np.any(pitches>0) else 0

    # Zero crossing rate
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=audio_data))

    # Spectral centroid
    spec_centroid = np.mean(librosa.feature.spectral_centroid(y=audio_data, sr=sr))

    return rms, pitch, zcr, spec_centroid

def estimate_voice_emotion(audio_data, sr=16000):
    rms, pitch, zcr, spec_centroid = extract_audio_features(audio_data, sr)

    # Rules (tuned for Jetson mic)
    if rms > 0.02 and pitch > 120 and zcr > 0.02:
        voice_tone = "happy/excited"
    elif rms < 0.01 and pitch < 100 and zcr < 0.01:
        voice_tone = "sad/calm"
    else:
        voice_tone = "neutral"

    return voice_tone

print(estimate_voice_emotion(audio_data))

🎤 Recording audio...
✅ Audio capture complete
Audio shape: (160000,)
neutral
